[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/electromagnetism/electrostriction/electrostriction.ipynb)

# Electrostriction in a Lattice of Induced Dipoles

The deformation of a dielectric in an applied electric field is called electrostriction. This example computes the coupled mechanical and electrical equilibrium of an elastic lattice of polarizable particles in a uniform field. The field induces a dipole moment on each particle, the dipoles attract head-to-tail and repel side-by-side against the restoring springs of the lattice, and shifting the particles changes the local field each dipole feels. Because reversing the applied field reverses every induced dipole without changing the forces between them, the equilibrium strain grows with the square of the field strength at weak fields.

Particle positions and induced dipoles are modelled as intrinsic spatial antivectors, while the electric field is modelled in spacetime as an extrinsic bivector that pairs with the worldsheet patch swept by a dipole along the observer's four-velocity via `field & (dipole & time)`. Taking the spacetime dual of that worldsheet map yields `field_along = (Point & time).dual()`, which maps any spatial direction to the electric field along it and forms both the single-site polarizability and the pairwise dipole-coupling operator. Assembling all particle pairs produces one coupled field map whose solve `dipole_response(positions, polarizability).solve(applied_field)` finds every equilibrium dipole at once. Keeping that solve inside the free energy lets automatic differentiation propagate dipole relaxation into both the force gradient and the mechanical Hessian for Newton steps `curvature.solve(gradient)`, following the variational induced-dipole formulation of [Simmonett et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC4545078/).

In Cartesian matrix-tensor notation, the induced moments solve the $2N \times 2N$ block system $\sum_j (\alpha^{-1}\delta_{ij}\mathbf I - \mathbf T_{ij})\boldsymbol\mu_j = \mathbf E_0$ with $\mathbf T_{ij} = (3\,\mathbf r_{ij}\mathbf r_{ij}^\mathsf{T} - r_{ij}^2\mathbf I)/r_{ij}^5$ for $j \ne i$, giving equilibrium electrostatic free energy $-\tfrac12\sum_i \mathbf E_0^\mathsf{T}\boldsymbol\mu_i$. Relaxing the lattice by Newton's method then assembles the $2N$-component force vector $F_{i\alpha} = \tfrac12\sum_{j \ne k}\boldsymbol\mu_j^\mathsf{T}(\partial\mathbf T_{jk}/\partial r_{i\alpha})\boldsymbol\mu_k - \partial U_{\mathrm{el}}/\partial r_{i\alpha}$ and the $2N \times 2N$ coordinate Hessian combining $\partial^2\mathbf T_{jk}/\partial r_{i\alpha}\partial r_{n\beta}$ with the implicit dipole-response term $(\partial\mathbf T/\partial\mathbf r)\,\mathbf A^{-1}(\partial\mathbf T/\partial\mathbf r)$.

In [ ]:
# In Colab, install numga and its examples from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[jax,examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline

from collections.abc import Iterator

import jax
import numpy as np

from numga import Algebra, stack
from numga.algebras import STA as ga
from numga.backend.jax import JaxContext, derivative
from numga.sparse import Sparse
from numga.topology import Topology

from examples.electromagnetism.electrostriction import render, scenarios
from examples.util.animation import save_figure, show_states

# Particles and dipoles lie in the xy-plane of spacetime `x+y+z+t-`, interacting via the
# three-dimensional electrostatic dipole field restricted to that plane with four pi times permittivity equal to one.
mv = JaxContext(ga).multivector
Scalar = ga.gatype.scalar()                                                    # Scalar+
Plane = ga.gatype.vector().derive.within(Algebra("x+y+"))                      # Vector+
# Particle positions, displacements, and induced dipoles are intrinsic spatial directions in the xy-plane;
# the electric field there is an extrinsic spacetime bivector with components xt and yt.
Point = Plane.derive.dual                                                      # Antivector-
Field = ga.gatype.bivector().derive.touching(Algebra("x+y+"), Algebra("t-"))   # Bivector+
Response = ga.gatype((Field, Point))                                           # Field+ <- Point-

# Observer four-velocity at rest with the lattice, pairing to one with `mv.t`.
time = -mv.t.dual()                                                            # Antivector-
# Spacetime duality map taking any spatial direction to the electric field along it.
field_along = (Point & time).dual()                                            # Field+ <- Point-

## Elastic reference lattice

Nearest neighbours in a hexagonal patch share harmonic springs, while weaker harmonic tethers to the initial sites prevent rigid drift and rotation without pinning the boundary. Applying the coboundary operator `topology.d[0]` maps particle positions to directed bond displacements across every edge.

In [ ]:
n_rings = 2
spacing = 1.0

In [ ]:
def lattice_positions(indices: np.ndarray, spacing: float) -> Point:
    """Integer axial coordinates (`[V, 2]`) map to reference lattice sites (`Point-[V]`) along two axes sixty degrees apart."""
    # Unit displacements along the two lattice directions, which meet at sixty degrees.
    first_axis = mv.x.dual()                                                   # [] Point-
    sixth_turn = (mv.xy * (-np.pi / 6)).exp()                                  # [] Rotor+
    axes = stack((first_axis, sixth_turn >> first_axis))                       # [2] Point-
    return (spacing * (axes * indices).sum(axis=-1)).field()


indices, bonds = scenarios.hexagonal_indices(n_rings)                          # [V, 2], [E, 2]
topology = Topology(len(indices), bonds)
V, E = topology.n_elements
rest_positions = lattice_positions(indices, spacing)                           # Point-[V]
bond_ends = topology.d[0]                                                      # Scalar+[E <- V]
save_figure(render.rest_figure(rest_positions, topology), "electrostriction_lattice");

## Induced dipoles on the rigid lattice

Each particle experiences both the external field and the three-dimensional electrostatic dipole fields of every other particle in the plane. Subtracting the pairwise dipole-coupling field map from the single-site inverse polarizability `field_along / polarizability` forms one coupled linear operator from dipole moments to applied fields.

In [ ]:
polarizability = 0.035 * spacing**3
field_strength = 3.05
dipole_scale = 4.0
field_scale = 0.65 / field_strength

In [ ]:
def dipole_response(positions: Point, polarizability: float) -> Response:
    """Single-site polarizability and pairwise dipole coupling form the linear field map (`[...] (Field+ <- Point-)[V <- V]`) from dipoles to applied fields."""
    sites = positions.batch()                                                  # [..., V] Point-
    n_particles = sites.shape[-1]
    separation = sites[..., :, None] - sites[..., None, :]                     # [..., V, V] Point-
    squared_distance = separation.squared()                                    # [..., V, V] Scalar+
    # Self pairs have a zero numerator, so adding one on the diagonal leaves them zero while keeping the denominator finite.
    distance = (squared_distance + np.eye(n_particles)).square_root()          # [..., V, V] Scalar+
    # Three times the radial field weighted by its pairing with the dipole's worldsheet patch, minus r^2 times the field along the dipole.
    separation_field = field_along(separation)                                 # [..., V, V] Field+
    dipole_law = 3 * separation_field * (separation_field & (Point & time)) - squared_distance * field_along
    coupling = (dipole_law / distance**5).field(0, 1)                          # [...] (Field+ <- Point-)[V <- V]
    return field_along / polarizability - coupling


applied_field = field_along(mv.x.dual()) * field_strength                      # [] Field+
rigid_dipoles = dipole_response(rest_positions, polarizability).solve(applied_field)  # Point-[V]
save_figure(
    render.lattice_figure(
        rest_positions, topology, rest_positions, rigid_dipoles, applied_field,
        dipole_scale, field_scale,
    ),
    "electrostriction_fixed",
);

Orange arrows show the induced dipole moments on the undeformed particles, aligned mainly with the gold applied-field arrow below the patch while tilting slightly near the boundary under mutual dipole fields.

## Coupled mechanical and electrical equilibrium

At fixed applied field, the electrostatic free energy after the dipoles equilibrate includes both the polarization self-energy and the work exchanged with the field source, reducing to minus half the pairing of the induced dipoles with the applied field. Keeping the dipole solve inside `total_energy` propagates dipole readjustment into both the first and second position derivatives during Newton relaxation.

In [ ]:
stiffness = 1.0
tether_stiffness = 0.15 * stiffness
n_newton_steps = 4

In [ ]:
def total_energy(
    positions: Point,
    rest_positions: Point,
    bond_ends: Sparse,
    applied_field: Field,
    polarizability: float,
    stiffness: float,
    tether_stiffness: float,
) -> Scalar:
    """Harmonic springs, weak site tethers, and equilibrated dipoles sum to the total free energy (`[...] Scalar+`) at fixed applied field."""
    rest_lengths = (bond_ends * rest_positions).antinorm()                     # Scalar+[E]
    extension = (bond_ends * positions).antinorm() - rest_lengths              # [...] Scalar+[E]
    bond_energy = stiffness / 2 * extension.squared().sites.sum()              # [...] Scalar+
    tether_energy = tether_stiffness / 2 * (positions - rest_positions).squared().sites.sum()
    # Solving for the dipoles inside the energy includes their readjustment in position derivatives.
    dipoles = dipole_response(positions, polarizability).solve(applied_field)  # [...] Point-[V]
    electric_energy = -(applied_field & (dipoles & time)).sites.sum() / 2      # [...] Scalar+
    return bond_energy + tether_energy + electric_energy


def mechanical_equilibrium(
    rest_positions: Point,
    bond_ends: Sparse,
    applied_field: Field,
    polarizability: float,
    stiffness: float,
    tether_stiffness: float,
    n_steps: int,
) -> Point:
    """Coupled Newton steps relax particle positions (`[...] Point-[V]`) from the reference lattice to electromechanical equilibrium."""
    @jax.jit
    def newton_step(positions: Point) -> Point:
        def energy(trial_positions: Point) -> Scalar:
            return total_energy(
                trial_positions, rest_positions, bond_ends, applied_field,
                polarizability, stiffness, tether_stiffness,
            )

        gradient = derivative(energy)(positions)                               # [...] (Scalar+ <- Point-)[V]
        curvature = derivative(derivative(energy))(positions)                  # [...] (Scalar+ <- (Point-, Point-))[V, V]
        return positions - curvature.solve(gradient)

    positions = rest_positions.broadcast_to(applied_field.shape)               # [...] Point-[V]
    for _ in range(n_steps):
        positions = newton_step(positions)
    return positions


equilibrium_positions = mechanical_equilibrium(
    rest_positions, bond_ends, applied_field,
    polarizability, stiffness, tether_stiffness, n_newton_steps,
)                                                                              # Point-[V]
equilibrium_dipoles = dipole_response(equilibrium_positions, polarizability).solve(applied_field)
save_figure(
    render.lattice_figure(
        rest_positions, topology, equilibrium_positions, equilibrium_dipoles, applied_field,
        dipole_scale, field_scale,
    ),
    "electrostriction_relaxed",
);

Solid blue bonds and markers show the relaxed lattice at true scale over the dashed grey reference lattice, contracting head-to-tail along the horizontal field where aligned dipoles attract and expanding slightly across it.

## Quadratic strain under increasing field

Reversing the applied field reverses every induced dipole and leaves the total free energy invariant, so the equilibrium strain grows quadratically at weak fields before stiffening as compressed bonds push back. Pairing centred particle positions with a reference plane measures the root-mean-square extent along that direction across a batch of field strengths.

In [ ]:
n_load_samples = 25

In [ ]:
def rms_extent(positions: Point, reference: Plane) -> Scalar:
    """Pairing centred particle offsets with a reference plane gives the root-mean-square extent (`[...] Scalar+`) along that direction."""
    offsets = positions - positions.sites.mean()                               # [...] Point-[V]
    return (reference & offsets).squared().sites.mean().square_root()


def axial_strain(positions: Point, rest_positions: Point, reference: Plane) -> Scalar:
    """Root-mean-square extent relative to the reference lattice gives the fractional axial strain (`[...] Scalar+`)."""
    return rms_extent(positions, reference) / rms_extent(rest_positions, reference) - 1


strengths = np.linspace(0, field_strength, n_load_samples)
load_fields = field_along(mv.x.dual()) * strengths                             # [n_load_samples] Field+
# Independent loading cases sit on the batch axis while particles inside each solve remain coupled.
load_positions = mechanical_equilibrium(
    rest_positions, bond_ends, load_fields,
    polarizability, stiffness, tether_stiffness, n_newton_steps,
)                                                                              # [n_load_samples] Point-[V]
parallel_strain = axial_strain(load_positions, rest_positions, mv.x)           # [n_load_samples] Scalar+
transverse_strain = axial_strain(load_positions, rest_positions, mv.y)         # [n_load_samples] Scalar+
save_figure(render.loading_figure(strengths, parallel_strain, transverse_strain), "electrostriction_loading");

Both curves depart tangentially from zero at weak fields, with the blue curve showing quadratic contraction along the applied field and the purple curve showing a smaller transverse expansion across it.

## Quasi-static deformation in a rotating field

Turning the full-strength field through one revolution traces a sequence of static electromechanical equilibria without inertia or dielectric lag. Each angle relaxes from the previous equilibrium, and because electrostriction is even in the field direction, the lattice returns to the same deformed shape after half a turn.

In [ ]:
n_frames = 126
frame_duration_ms = 40

In [ ]:
def turning_states(
    rest_positions: Point,
    bond_ends: Sparse,
    field_strength: float,
    polarizability: float,
    stiffness: float,
    tether_stiffness: float,
    n_frames: int,
    n_steps: int,
) -> Iterator[tuple[Point, Point, Field]]:
    """Yields equilibrium positions, induced dipoles, and applied field (`Point-[V]`, `Point-[V]`, `Field+`) across one revolution."""
    @jax.jit
    def frame_equilibrium(positions: Point, field: Field) -> tuple[Point, Point]:
        def energy(trial_positions: Point) -> Scalar:
            return total_energy(
                trial_positions, rest_positions, bond_ends, field,
                polarizability, stiffness, tether_stiffness,
            )

        for _ in range(n_steps):
            gradient = derivative(energy)(positions)                           # (Scalar+ <- Point-)[V]
            curvature = derivative(derivative(energy))(positions)              # (Scalar+ <- (Point-, Point-))[V, V]
            positions = positions - curvature.solve(gradient)
        dipoles = dipole_response(positions, polarizability).solve(field)      # Point-[V]
        return positions, dipoles

    angles = np.linspace(0, 2 * np.pi, n_frames, endpoint=False)
    turns = (mv.xy * (-angles / 2)).exp()                                      # [n_frames] Rotor+
    fields = field_along(turns >> mv.x.dual()) * field_strength                # [n_frames] Field+
    positions = rest_positions                                                 # Point-[V]
    for field in fields:
        positions, dipoles = frame_equilibrium(positions, field)
        yield positions, dipoles, field


cycle_states = show_states(
    turning_states(
        rest_positions, bond_ends, field_strength,
        polarizability, stiffness, tether_stiffness, n_frames, n_newton_steps,
    ),
    render.lattice_animation,
    rest_positions, topology, dipole_scale, field_scale,
    name="electrostriction",
    duration_ms=frame_duration_ms,
)

As the gold applied-field arrow rotates once around, the orange induced dipoles turn once while the blue compressed axis of the lattice completes two cycles of contraction over the dashed grey reference patch.

In [ ]:
# checks
assert ((load_positions[0] - rest_positions).antinorm() < 1e-6).all()
assert parallel_strain[-1] < 0
assert transverse_strain[-1] > 0
assert (parallel_strain[2] / parallel_strain[1] - 4).abs() < 0.05
initial_shape, initial_dipoles, _ = cycle_states[0]
opposite_shape, opposite_dipoles, _ = cycle_states[n_frames // 2]
assert ((opposite_shape - initial_shape).antinorm() < 1e-3).all()
assert ((opposite_dipoles + initial_dipoles).antinorm() < 1e-3).all()